# 🍎 Health Calculator Agent Tutorial 🍏

Welcome to the **Health Calculator Agent** tutorial, where we'll showcase how to:
1. **Initialize** a project and use the Microsoft Foundry ecosystem
2. **Create an Agent** with **Code Interpreter** capabilities
3. **Perform BMI calculations** and **analyze nutritional data** with sample CSV files
4. **Generate** basic health insights and disclaimers

> #### Ensure you have completed the [`1-basics.ipynb`](./1-basics.ipynb) notebook before starting this one.

## Let's Dive In
We'll walk step-by-step, similar to our **Fun & Fit** sample, but with a focus on using **Code Interpreter** for numeric calculations and data analysis. Let's begin!

<img src="./seq-diagrams/2-code-interpreter.png" width="30%"/>




## 🔐 Authentication Setup

Before running the next cell, make sure you're authenticated with Azure CLI. 

* Open a terminal inside VSC (Visual Studio Code).
    * Run the following command in your terminal:

```
az login --use-device-code
```

* This will provide you with a device code and URL to authenticate in your browser to Azure.
    * Authenticate using the skillable Azure **Username** and **TAP**(Temporary Access Pass).
* Go back to the terminal and select the **default subscription.**

The Device Token will be used in this lab for:

* Remote development environments
* Systems without a default browser
* Corporate environments with strict security policies

* After successful authentication, you can proceed with the notebook cells below.

## 1. Initial Setup
We'll start by importing libraries, loading environment variables, and initializing an **AIProjectClient**. We'll also create a sample CSV for demonstration.


In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AutoCodeInterpreterToolParam,
    CodeInterpreterTool,
    PromptAgentDefinition,
)

env_path = next(
    (directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError(
        "Could not find .env. Complete Lab 00 and place it in the repository root."
    )

load_dotenv(env_path)
tenant_id = os.environ.get("TENANT_ID")
ai_foundry_project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.environ.get("MODEL_DEPLOYMENT_NAME")
missing_variables = [
    name
    for name, value in {
        "TENANT_ID": tenant_id,
        "AI_FOUNDRY_PROJECT_ENDPOINT": ai_foundry_project_endpoint,
        "MODEL_DEPLOYMENT_NAME": model_deployment_name,
    }.items()
    if not value
]
if missing_variables:
    raise ValueError(f"Missing required .env variables: {', '.join(missing_variables)}")

credential = AzureCliCredential(tenant_id=tenant_id)
project_client = AIProjectClient(
    endpoint=ai_foundry_project_endpoint,
    credential=credential,
)
openai_client = project_client.get_openai_client()
print(f"📁 Environment loaded from: {env_path}")
print("✅ Successfully initialized AIProjectClient and OpenAI client")


def create_sample_data():
    data = {
        "Date": pd.date_range(start="2024-01-01", periods=7),
        "Calories": [2100, 1950, 2300, 2050, 1900, 2200, 2150],
        "Protein_g": [80, 75, 85, 78, 72, 82, 79],
        "Carbs_g": [250, 230, 270, 245, 225, 260, 255],
        "Fat_g": [70, 65, 75, 68, 63, 73, 71],
        "Fiber_g": [25, 22, 28, 24, 21, 26, 23],
    }
    filename = os.environ.get("NUTRITION_DATA_FILENAME", "nutrition_data.csv")
    pd.DataFrame(data).to_csv(filename, index=False)
    print(f"📄 Created sample data file: {filename}")
    return filename


sample_file = create_sample_data()

## 2. Create Health Calculator Agent 👩‍💻
We'll upload our sample CSV and then create an agent with **Code Interpreter** enabled. This agent can read the file, run Python code, and return results and visualizations.


In [ ]:
def create_health_calculator(file_path):
    """Upload nutrition data and create a Code Interpreter prompt agent."""
    print(f"📤 Uploading file: {file_path}")
    with open(file_path, "rb") as data_file:
        uploaded_file = openai_client.files.create(
            purpose="assistants",
            file=data_file,
        )
    print(f"✅ Uploaded CSV file, ID: {uploaded_file.id}")

    code_interpreter = CodeInterpreterTool(
        container=AutoCodeInterpreterToolParam(file_ids=[uploaded_file.id])
    )
    agent = project_client.agents.create_version(
        agent_name="health-calculator-agent",
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions="""
            You are a health calculator agent that can:
            1. Calculate and interpret BMI using BMI = weight(kg) / height(m)².
            2. Analyze the uploaded nutrition CSV.
            3. Generate charts and plots for data visualization.
            4. Use Code Interpreter for calculations and visualizations.
            5. State that results are educational, not medical advice, and encourage
               consultation with a healthcare professional.
            """,
            tools=[code_interpreter],
        ),
    )
    print(f"🎉 Created agent {agent.name}, version: {agent.version}")
    return agent, uploaded_file


health_agent, uploaded_file = create_health_calculator(sample_file)

## 3. BMI Calculation with Code Interpreter

Create a Conversation and ask the agent to calculate BMI with Python. Code Interpreter is available to the prompt agent for the entire response.

In [ ]:
def calculate_bmi_with_agent(agent, height_inches, weight_pounds):
    conversation = openai_client.conversations.create()
    user_text = (
        f"Calculate BMI for a height of {height_inches} inches and a weight of "
        f"{weight_pounds} pounds. Show the unit conversions and Python calculation, "
        "interpret the result, and include an appropriate health disclaimer."
    )
    response = openai_client.responses.create(
        conversation=conversation.id,
        input=user_text,
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "type": "agent_reference",
            }
        },
    )
    print(f"🤖 BMI response created, ID: {response.id}")
    return conversation, response


bmi_conversation, bmi_response = calculate_bmi_with_agent(health_agent, 70, 180)

## 4. Nutrition Analysis

The uploaded CSV is configured on `CodeInterpreterTool`, so a new Conversation can ask the agent to load the file, calculate weekly averages, and generate a chart.

In [ ]:
def analyze_nutrition_data(agent):
    conversation = openai_client.conversations.create()
    response = openai_client.responses.create(
        conversation=conversation.id,
        input=(
            "Analyze the uploaded nutrition CSV. Compute average daily calories, "
            "protein, carbohydrates, fat, and fiber; identify useful patterns; and "
            "create a clearly labeled weekly trend chart. Include a health disclaimer."
        ),
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "type": "agent_reference",
            }
        },
    )
    print(f"🤖 Nutrition response created, ID: {response.id}")
    return conversation, response


nutrition_conversation, nutrition_response = analyze_nutrition_data(health_agent)

## 5. Viewing Results & Visualizations 📊

Responses expose generated files through `container_file_citation` annotations. Use the cited container and file IDs to download each generated chart.

In [ ]:
def view_agent_response(label, response):
    """Print response text and download generated container files."""
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)
    print(response.output_text)

    downloaded = set()
    for item in response.output:
        if item.type != "message":
            continue
        for block in item.content:
            if block.type != "output_text":
                continue
            for annotation in block.annotations:
                if annotation.type != "container_file_citation":
                    continue
                key = (annotation.container_id, annotation.file_id)
                if key in downloaded:
                    continue
                downloaded.add(key)
                file_content = openai_client.containers.files.content.retrieve(
                    file_id=annotation.file_id,
                    container_id=annotation.container_id,
                )
                file_content.write_to_file(annotation.filename)
                print(f"💾 Downloaded generated file: {annotation.filename}")


view_agent_response("🧮 BMI CALCULATION RESULTS", bmi_response)
view_agent_response("📊 NUTRITION ANALYSIS RESULTS", nutrition_response)

## 6. Cleanup & Best Practices
We can remove our agent and sample data if desired. In production, you might keep them for repeated usage.

### Best Practices in a Nutshell
1. **Data Handling** – Validate input data, handle missing values, properly manage file attachments.
2. **Calculations** – Provide formula steps, disclaimers, limit scope to general wellness, remind user you're not a doctor.
3. **Visualizations** – Use clear labeling and disclaimers that charts are for educational demonstrations.
4. **Security** – Monitor usage, limit access to code interpreter if dealing with proprietary data.


In [ ]:
def cleanup_all():
    for conversation in (bmi_conversation, nutrition_conversation):
        openai_client.conversations.delete(conversation_id=conversation.id)
    print("🗑️ Deleted Conversations.")

    project_client.agents.delete_version(
        agent_name=health_agent.name,
        agent_version=health_agent.version,
    )
    print("🗑️ Deleted health calculator agent version.")

    openai_client.files.delete(uploaded_file.id)
    print("🗑️ Deleted uploaded CSV file.")

    if sample_file and os.path.exists(sample_file):
        os.remove(sample_file)
        print("🗑️ Deleted local sample CSV file.")

    openai_client.close()
    project_client.close()
    credential.close()
    print("✅ Cleanup completed!")


cleanup_all()

# Congratulations! 🎉

You uploaded a CSV with `openai_client.files.create()`, configured it on `CodeInterpreterTool` with `AutoCodeInterpreterToolParam`, invoked a versioned prompt agent through Responses, and downloaded generated files from their `container_file_citation` annotations. Keep the health and privacy disclaimers when adapting this pattern to real data.